# Inspect the held-out test data with frozen models

This companion loads the existing checkpoints and runs **inference only**. Run it directly; the three numbered training notebooks do not need to be rerun. There are no optimizer steps, fitting, checkpoint writes or changes to model selection.

The original test split has **732 eligible EPL fixtures, including 75 Arsenal fixtures**, from 2024/25–2025/26. It was held out from training and checkpoint selection, but its results have already been inspected: this is a reproducibility check, not a newly untouched test.

## Where the files are

- [Fixture identity and dates](../arsenal_predictor_v1/data/test/index.csv)
- [39 input features](../arsenal_predictor_v1/data/test/X.csv)
- [Actual outcomes and scores](../arsenal_predictor_v1/data/test/y.csv)
- [Saved models](../results/current/models/), including `arsenal_mlp_best.pt`, `league_mlp_best.pt`, `league_to_arsenal_mlp_best.pt` and the validation-selected `league_logistic.npz`.

Rows of the three test CSVs align by position; the frozen manifest hashes verify their identity. `A/D/H` means away win/draw/home win. Arsenal tables reorient this into **Win/Draw/Loss for Arsenal**, including away fixtures.

The models and metrics are fixed. Use the full fixture tables to assess errors; do not choose a new model on this test set and then call it an independent validation. Tables, rather than new plots, are used here to make every fixture and exact probability inspectable.

## 1. Load the existing artifacts

In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import hashlib, json, platform
import numpy as np
import pandas as pd
import torch
from IPython.display import display, Markdown

ROOT=Path.cwd()
if not (ROOT/'arsenal_predictor_v1').exists():
    ROOT=ROOT.parent
assert (ROOT/'arsenal_predictor_v1/data/test/X.csv').exists(), 'Run from the repository root or notebooks/.'
import sys
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from experiments.inference import predict
from experiments.train import score

DATA=ROOT/'arsenal_predictor_v1'
FROZEN=ROOT/'results/current'
MODELS=FROZEN/'models'
OUT=ROOT/'results/test_review'
OUT.mkdir(parents=True,exist_ok=True)
started=datetime.now(ZoneInfo('Asia/Shanghai')).isoformat()
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
# Preserve every existing study artifact and all frozen input files.
protected={str(p.relative_to(ROOT)):sha(p) for base in [DATA,FROZEN] for p in base.rglob('*') if p.is_file()}
manifest=json.loads((FROZEN/'manifest.json').read_text())
for path,digest in manifest['input_sha256'].items(): assert sha(DATA/path)==digest,path
for path,digest in manifest['code_sha256'].items(): assert sha(ROOT/path)==digest,path
expected=json.loads((FROZEN/'artifact_checksums.json').read_text())['files']
for path,digest in expected.items(): assert sha(ROOT/path)==digest,path
selection=json.loads((MODELS/'selected_model.json').read_text())
selected=selection['selected_model']
model_names=list(selection['validation_scores'])
model_manifest=json.loads((MODELS/'run_manifest.json').read_text())
torch.set_num_threads(manifest['config']['threads'])
torch.use_deterministic_algorithms(True)
print('Frozen run:',manifest['run_id'])
print('Validation-selected model:',selected)
print('Artifacts:',MODELS.relative_to(ROOT))
print('No fitting or model selection will be performed.')

Frozen run: e3ceec90-4b79-4a9f-a6dc-b2dd43e67546
Validation-selected model: league_logistic
Artifacts: results/current/models
No fitting or model selection will be performed.


## 2. Inspect the actual held-out fixtures
Features are passed to inference without targets or score columns. The rows below include outcomes only so you can judge the predictions. Excluded fixtures with incomplete lagged features are not part of this eligible test split.

In [2]:
features=pd.read_csv(DATA/'data/test/X.csv').astype(np.float32)
fixtures=pd.read_csv(DATA/'data/test/index.csv')
targets=pd.read_csv(DATA/'data/test/y.csv')
assert len(features)==len(fixtures)==len(targets)==732
assert list(features)==model_manifest['features']
assert fixtures.match_id.is_unique and fixtures['split'].eq('test').all()
assert np.isfinite(features.to_numpy()).all()
for split in ['train','validation']:
    earlier=pd.read_csv(DATA/f'data/{split}/index.csv')
    assert not set(fixtures.match_id)&set(earlier.match_id)
    assert earlier.date.max()<fixtures.date.min()
classes=model_manifest['classes']
assert classes==['A','D','H']
y=targets.target_result.map({name:i for i,name in enumerate(classes)}).to_numpy()
assert not pd.isna(y).any()
arsenal=(fixtures.home_team.eq('Arsenal')|fixtures.away_team.eq('Arsenal')).to_numpy()
assert arsenal.sum()==75
actual=pd.concat([fixtures,targets],axis=1)
actual['score']=actual.target_home_goals.astype(str)+'–'+actual.target_away_goals.astype(str)
actual.to_csv(OUT/'held_out_test_fixtures.csv',index=False)
actual.loc[arsenal].to_csv(OUT/'arsenal_test_fixtures.csv',index=False)
print('Held-out dates:',fixtures.date.min(),'to',fixtures.date.max())
print('EPL fixtures:',len(fixtures),'| Arsenal fixtures:',int(arsenal.sum()),'| Features:',features.shape[1])
display(actual.groupby('season_start').agg(fixtures=('match_id','size'),first_date=('date','min'),last_date=('date','max')))
display(actual.loc[arsenal,['date','home_team','away_team','score','target_result']].head(10))
print('The exports contain every fixture, not just this preview.')

Held-out dates: 2024-08-16 to 2026-05-24
EPL fixtures: 732 | Arsenal fixtures: 75 | Features: 39


,fixtures,first_date,last_date
season_start,,,
2024,366,2024-08-16,2025-05-25
2025,366,2025-08-15,2026-05-24


,date,home_team,away_team,score,target_result
1,2024-08-17,Arsenal,Wolves,2–0,H
7,2024-08-24,Aston Villa,Arsenal,0–2,A
17,2024-08-31,Arsenal,Brighton,1–1,D
26,2024-09-15,Tottenham,Arsenal,0–1,A
35,2024-09-22,Manchester City,Arsenal,2–2,D
38,2024-09-28,Arsenal,Leicester,4–2,H
47,2024-10-05,Arsenal,Southampton,3–1,H
61,2024-10-19,Bournemouth,Arsenal,2–0,H
75,2024-10-27,Arsenal,Liverpool,2–2,D
80,2024-11-02,Newcastle,Arsenal,1–0,H


The exports contain every fixture, not just this preview.


## 3. Load frozen models and predict

`predict(MODELS, features, name)` reconstructs the saved neural architecture, loads checkpoint weights/scaler and uses `eval()` with `torch.no_grad()`. For logistic models it loads saved scaler parameters and coefficients from `.npz`; no estimator is fitted. Targets are used only after predictions to calculate metrics. All six original arms are retained; test rankings do not change the validation-selected candidate.

In [3]:
predictions={}
records=[]
for name in model_names:
    probabilities=predict(MODELS,features,name)
    assert probabilities.shape==(732,3)
    assert list(probabilities)==classes
    values=probabilities.to_numpy()
    assert np.isfinite(values).all() and (values>=0).all() and (values<=1).all()
    np.testing.assert_allclose(values.sum(axis=1),1,atol=1e-6)
    predictions[name]=probabilities
    for population,mask in [('Arsenal',arsenal),('All EPL',np.ones(len(y),dtype=bool))]:
        records.append({'model':name,'population':population,**score(y[mask],values[mask])})
metrics=pd.DataFrame(records)
metrics.to_csv(OUT/'metrics.csv',index=False)
print('Arsenal held-out test: lower log loss and Brier are better; higher accuracy is better.')
display(metrics.loc[metrics.population.eq('Arsenal')].round(6))
# Reconcile only after independently loading models and computing new predictions.
previous=pd.read_csv(FROZEN/'evaluation/metrics.csv')
for row in metrics.itertuples():
    old=previous.query('split=="test" and model==@row.model and population==@row.population').iloc[0]
    for metric in ['log_loss','brier_multiclass','accuracy']:
        np.testing.assert_allclose(getattr(row,metric),old[metric],rtol=0,atol=1e-12)
print('Reloaded model metrics agree with the frozen evaluation.')

Arsenal held-out test: lower log loss and Brier are better; higher accuracy is better.


,model,population,log_loss,brier_multiclass,accuracy,n
0,arsenal_mlp,Arsenal,0.959723,0.569044,0.600000,75
2,league_mlp,Arsenal,0.899941,0.529087,0.613333,75
4,league_to_arsenal_mlp,Arsenal,0.900410,0.529357,0.613333,75
6,league_logistic,Arsenal,0.908957,0.532644,0.613333,75
8,arsenal_logistic,Arsenal,0.909300,0.534445,0.640000,75
10,league_prior,Arsenal,1.094381,0.664183,0.400000,75


Reloaded model metrics agree with the frozen evaluation.


## 4. Every Arsenal fixture, actual outcome and predicted probabilities

The main table uses the validation-selected candidate. Every model gets its own complete CSV and readable Markdown table under [`results/test_review/`](../results/test_review/README.md). `Correct` uses the largest predicted probability; `Match log loss` also penalizes incorrect confidence. A forecast of 40% win is not a promise of a win.

An additional table compares all three neural models' predicted outcomes on the same fixtures.

In [4]:
tables={}
base=actual.loc[arsenal].reset_index(drop=True)
home=base.home_team.eq('Arsenal').to_numpy()
result=base.target_result.to_numpy()
actual_arsenal=np.where(result=='D','Draw',np.where((home&(result=='H'))|(~home&(result=='A')),'Win','Loss'))
for name,probs in predictions.items():
    table=actual.copy()
    for label in classes: table['p_'+label]=probs[label].to_numpy()
    table['predicted_result']=np.array(classes)[probs.to_numpy().argmax(axis=1)]
    table.to_csv(OUT/f'{name}_all_test_predictions.csv',index=False)
    p=probs.loc[arsenal].reset_index(drop=True)
    oriented=np.column_stack([np.where(home,p.H,p.A),p.D,np.where(home,p.A,p.H)])
    review=pd.DataFrame({'match_id':base.match_id,'date':base.date,'opponent':np.where(home,base.away_team,base.home_team),'venue':np.where(home,'Home','Away'),'score_home_away':base.score,'actual_arsenal':actual_arsenal,'predicted_arsenal':np.array(['Win','Draw','Loss'])[oriented.argmax(axis=1)],'p_win':oriented[:,0],'p_draw':oriented[:,1],'p_loss':oriented[:,2]})
    review['correct']=review.actual_arsenal.eq(review.predicted_arsenal)
    review['match_log_loss']=-np.log(np.clip(p.to_numpy()[np.arange(len(p)),y[arsenal]],1e-12,1))
    review.to_csv(OUT/f'{name}_arsenal_test_review.csv',index=False)
    lines=[f'# {name}: all 75 held-out Arsenal fixtures','',f'Frozen run: `{manifest["run_id"]}`. Score is home–away, even when Arsenal are away. Probabilities below are percentages.','', '| Date | Opponent | Venue | Score H–A | Actual | Predicted | Win % | Draw % | Loss % | Correct | Log loss |','|---|---|---|---|---|---|---:|---:|---:|---|---:|']
    for row in review.itertuples():
        lines.append(f'| {row.date} | {row.opponent} | {row.venue} | {row.score_home_away} | {row.actual_arsenal} | {row.predicted_arsenal} | {100*row.p_win:.2f} | {100*row.p_draw:.2f} | {100*row.p_loss:.2f} | {"Yes" if row.correct else "No"} | {row.match_log_loss:.4f} |')
    (OUT/f'{name}_arsenal_test_review.md').write_text(chr(10).join(lines)+chr(10))
    tables[name]=review
selected_table=tables[selected].drop(columns='match_id')
for season in [2024,2025]:
    mask=base.season_start.eq(season)
    display(Markdown(f'### {season}/{str(season+1)[-2:]} — {selected}'))
    with pd.option_context('display.max_rows',80,'display.max_columns',20,'display.float_format','{:.4f}'.format):
        display(selected_table.loc[mask])
comparison=tables[selected][['date','opponent','venue','actual_arsenal']].copy()
for name in ['arsenal_mlp','league_mlp','league_to_arsenal_mlp']:
    comparison[name]=tables[name].predicted_arsenal
comparison.to_csv(OUT/'neural_model_fixture_comparison.csv',index=False)
display(Markdown('### Neural model outcomes: first ten fixtures (all 75 exported)'))
display(comparison.head(10))

### 2024/25 — league_logistic

,date,opponent,venue,score_home_away,actual_arsenal,predicted_arsenal,p_win,p_draw,p_loss,correct,match_log_loss
0,2024-08-17,Wolves,Home,2–0,Win,Win,0.7624,0.1711,0.0665,True,0.2713
1,2024-08-24,Aston Villa,Away,0–2,Win,Win,0.5499,0.2271,0.2230,True,0.5980
2,2024-08-31,Brighton,Home,1–1,Draw,Win,0.7122,0.1786,0.1091,False,1.7225
3,2024-09-15,Tottenham,Away,0–1,Win,Win,0.3839,0.2554,0.3607,True,0.9574
4,2024-09-22,Manchester City,Away,2–2,Draw,Loss,0.1979,0.1850,0.6171,False,1.6874
5,2024-09-28,Leicester,Home,4–2,Win,Win,0.7283,0.2003,0.0714,True,0.3170
6,2024-10-05,Southampton,Home,3–1,Win,Win,0.7766,0.1695,0.0539,True,0.2528
7,2024-10-19,Bournemouth,Away,2–0,Loss,Win,0.5797,0.2172,0.2031,False,1.5942
8,2024-10-27,Liverpool,Home,2–2,Draw,Win,0.4421,0.2459,0.3119,False,1.4027
9,2024-11-02,Newcastle,Away,1–0,Loss,Win,0.4982,0.2172,0.2846,False,1.2565


### 2025/26 — league_logistic

,date,opponent,venue,score_home_away,actual_arsenal,predicted_arsenal,p_win,p_draw,p_loss,correct,match_log_loss
38,2025-08-17,Manchester United,Away,0–1,Win,Win,0.5157,0.2344,0.2499,True,0.6622
39,2025-08-31,Liverpool,Away,1–0,Loss,Loss,0.2929,0.2209,0.4862,True,0.7211
40,2025-09-13,Nottingham Forest,Home,3–0,Win,Win,0.6742,0.1936,0.1322,True,0.3942
41,2025-09-21,Manchester City,Home,1–1,Draw,Win,0.5337,0.2098,0.2564,False,1.5615
42,2025-09-28,Newcastle,Away,1–2,Win,Loss,0.3545,0.2488,0.3968,False,1.0371
43,2025-10-04,West Ham,Home,2–0,Win,Win,0.7590,0.1717,0.0693,True,0.2758
44,2025-10-18,Fulham,Away,0–1,Win,Win,0.5816,0.2361,0.1823,True,0.5420
45,2025-10-26,Crystal Palace,Home,1–0,Win,Win,0.6689,0.1872,0.1439,True,0.4021
46,2025-11-01,Burnley,Away,0–2,Win,Win,0.6997,0.1931,0.1072,True,0.3571
47,2025-11-08,Sunderland,Away,2–2,Draw,Win,0.6435,0.1994,0.1571,False,1.6126


### Neural model outcomes: first ten fixtures (all 75 exported)

,date,opponent,venue,actual_arsenal,arsenal_mlp,league_mlp,league_to_arsenal_mlp
0,2024-08-17,Wolves,Home,Win,Win,Win,Win
1,2024-08-24,Aston Villa,Away,Win,Win,Win,Win
2,2024-08-31,Brighton,Home,Draw,Win,Win,Win
3,2024-09-15,Tottenham,Away,Win,Loss,Loss,Loss
4,2024-09-22,Manchester City,Away,Draw,Loss,Loss,Loss
5,2024-09-28,Leicester,Home,Win,Win,Win,Win
6,2024-10-05,Southampton,Home,Win,Win,Win,Win
7,2024-10-19,Bournemouth,Away,Loss,Win,Win,Win
8,2024-10-27,Liverpool,Home,Draw,Win,Win,Win
9,2024-11-02,Newcastle,Away,Loss,Win,Win,Win


## 5. Record the test-only execution and verify preservation

In [5]:
unchanged=all(sha(ROOT/path)==digest for path,digest in protected.items())
assert unchanged, 'An existing input or study artifact changed.'
assert selection==json.loads((MODELS/'selected_model.json').read_text())
for path,digest in expected.items(): assert sha(ROOT/path)==digest,path
record={'mode':'frozen-model inference only; previously inspected held-out test','started_at_shanghai':started,'finished_at_shanghai':datetime.now(ZoneInfo('Asia/Shanghai')).isoformat(),'frozen_run_id':manifest['run_id'],'selected_model_unchanged':selected,'models_scored':model_names,'test_rows':len(features),'arsenal_test_rows':int(arsenal.sum()),'training_performed':False,'original_inputs_and_study_artifacts_unchanged':unchanged,'protected_file_count':len(protected),'input_sha256':{f'data/test/{name}.csv':sha(DATA/f'data/test/{name}.csv') for name in ['X','y','index']},'model_sha256':{p.name:sha(p) for p in MODELS.iterdir() if p.suffix in ['.pt','.npz','.npy']},'python':platform.python_version(),'torch':torch.__version__,'numpy':np.__version__,'metrics_match_frozen_evaluation':True}
record['export_sha256']={p.name:sha(p) for p in OUT.iterdir() if p.suffix in ['.csv','.md'] and p.name!='README.md'}
(OUT/'execution.json').write_text(json.dumps(record,indent=2)+chr(10))
links=[f'- [{name}: all 75 Arsenal fixtures]({name}_arsenal_test_review.md) · [CSV]({name}_arsenal_test_review.csv)' for name in model_names]
readme=['# Held-out test review — frozen models','', 'Inference-only companion to [the executed notebook](../../notebooks/Frozen_Model_Test_Review.ipynb). No retraining, tuning, checkpoint changes or new selection. The test period was held out from fitting but previously inspected.','',f'Frozen run: `{manifest["run_id"]}`. Candidate selected on validation: **{selected}**.','', '[All 732 eligible test fixtures and actual scores](held_out_test_fixtures.csv) · [75 Arsenal fixtures](arsenal_test_fixtures.csv) · [Metrics](metrics.csv) · [Three neural predictions side by side](neural_model_fixture_comparison.csv)','', 'Original test files: [features](../../arsenal_predictor_v1/data/test/X.csv), [outcomes](../../arsenal_predictor_v1/data/test/y.csv), [fixture index](../../arsenal_predictor_v1/data/test/index.csv). Rows align by position. A/D/H means away win/draw/home win; the review tables use Arsenal Win/Draw/Loss.','',*links,'','Each model also has a complete `*_all_test_predictions.csv` containing all 732 fixtures and A/D/H probabilities. Model files are loaded from [current/models/](../current/models/).','', '[Execution and preservation checks](execution.json). Tables are recomputed when this notebook runs; original inputs and current-study artifacts are verified unchanged.']
(OUT/'README.md').write_text(chr(10).join(readme)+chr(10))
print('No training performed. Original inputs, selection, checkpoints and study outputs are unchanged.')
print('Selected candidate correct:',int(tables[selected].correct.sum()),'of',int(arsenal.sum()))
print('Review files:',OUT.relative_to(ROOT))

No training performed. Original inputs, selection, checkpoints and study outputs are unchanged.
Selected candidate correct: 46 of 75
Review files: results/test_review


## Reading the verdict

Use log loss as the primary probability-quality measure and the fixture tables to inspect confidence and recurring errors. Accuracy alone ignores probability calibration. The original validation-selected candidate remains `league_logistic`; it is not replaced using these test results. Neural-arm comparisons retain the same data, learned weights and feature preparation as the saved study.

Repeating inference on the same frozen model and inputs should reproduce its scores. This notebook provides an inspectable test ledger; a new generalization claim would require an untouched later dataset.